In [1]:
import csv
from datetime import datetime

# Step 1: Load weather data into a dictionary
weather_by_datetime = {}

with open("hourly_weather.csv", "r") as wf:
    reader = csv.DictReader(wf)
    for row in reader:
        weather_by_datetime[row["datetime"]] = {
            "temperature_2m": row["temperature_2m"],
            "precipitation": row["precipitation"],
            "wind_speed_10m": row["wind_speed_10m"]
        }

# Step 2: Stream 311 data and write merged results
with open("311_Service_Requests.csv", "r") as f311, \
     open("311_weather_merged1.csv", "w", newline="") as fout:

    reader = csv.DictReader(f311)
    fieldnames = reader.fieldnames + ["datetime", "temperature_2m", "precipitation", "wind_speed_10m"]
    writer = csv.DictWriter(fout, fieldnames=fieldnames)
    writer.writeheader()

    for row in reader:
        try:
            created_date = datetime.strptime(row["Created Date"], "%m/%d/%Y %I:%M:%S %p")
            dt_hour = created_date.replace(minute=0, second=0, microsecond=0).strftime("%Y-%m-%dT%H:00")
            if dt_hour in weather_by_datetime:
                row["datetime"] = dt_hour
                row.update(weather_by_datetime[dt_hour])
                writer.writerow(row)
        except Exception as e:
            continue  # skip bad rows
